## Work with GPU

In [34]:
import torch
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
import optuna

In [35]:
torch.cuda.is_available()

True

In [36]:
import torch

print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

2.11.0+cu128
12.8
True
NVIDIA GeForce RTX 4050 Laptop GPU


In [37]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device being used:',device)

Device being used: cuda


In [38]:
device

device(type='cuda')

In [39]:
df = pd.read_csv("fashion-mnist_train.csv")
df.shape

(60000, 785)

In [40]:
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [41]:
features = df.iloc[:,1:]
targets = df.iloc[:,0]
targets

0        2
1        9
2        6
3        0
4        3
        ..
59995    9
59996    1
59997    8
59998    8
59999    7
Name: label, Length: 60000, dtype: int64

In [42]:
x_train, x_test, y_train, y_test = train_test_split(features, targets, test_size=0.2, random_state= 42)
x_train /= 255.0
x_test /= 255.0

In [43]:
x_train.shape

(48000, 784)

In [44]:
x_test.shape

(12000, 784)

In [45]:
device

device(type='cuda')

In [46]:
# to create the dataset
class CustomDataset(Dataset):
    def __init__(self, features, targets):
        self.features = torch.tensor(features.to_numpy()).to(torch.float32)
        self.targets = torch.tensor(targets.to_numpy()).to(torch.long)
    
    def __len__(self):
        return self.features.shape[0]
    
    def __getitem__(self, index):
        return (self.features[index], self.targets[index])

In [47]:
# dataset collection stored here
train_dataset = CustomDataset(x_train, y_train)
test_dataset = CustomDataset(x_test, y_test)

In [48]:
train_dataset.__len__()

48000

In [49]:
test_dataset.__len__()

12000

In [50]:
train_dataset.__getitem__(0)

(tensor([0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.2275,
         0.5333, 0.0000, 0.0

In [51]:
train_dataset_loader = DataLoader(train_dataset, batch_size= 32, shuffle= True, pin_memory=True)
test_dataset_loader = DataLoader(test_dataset, batch_size= 32, shuffle=False, pin_memory=True)

In [52]:
for feature, target in train_dataset_loader:
    print(feature)
    print(target)
    break

tensor([[0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        ...,
        [0.0000, 0.0000, 0.0000,  ..., 0.1529, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000]])
tensor([2, 8, 1, 7, 1, 7, 6, 8, 2, 1, 6, 9, 1, 9, 1, 7, 5, 4, 8, 2, 0, 6, 9, 6,
        0, 1, 2, 8, 3, 2, 2, 3])


In [53]:
class ImageClassifier(nn.Module):

    def __init__(self, number_of_features):
        super().__init__()

        # Image classifier model
        self.model = nn.Sequential(
            nn.Linear(number_of_features, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(p = 0.4),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(p = 0.4),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(p = 0.4),
            nn.Linear(32, 10)
        )

        # optimizer
        self.optim = optim.SGD(self.model.parameters(), lr = 0.1, weight_decay= 1e-4)
    
    def forward(self, features):
        self.y_pred = self.model(features)
        return self.y_pred
    
    def loss_function(self, y_real):
        loss_fn = nn.CrossEntropyLoss()
        self.loss = loss_fn(self.y_pred, y_real)
        return self.loss.item()
    
    def optimization(self):
        self.optim.zero_grad()
        self.loss.backward()
        self.optim.step()


In [54]:
x_train.shape[1]

784

In [55]:
# model is defined but the model uses the "cpu"
model = ImageClassifier(x_train.shape[1])

In [56]:
for i in model.named_parameters():
    print(i)

('model.0.weight', Parameter containing:
tensor([[ 0.0342, -0.0153,  0.0011,  ...,  0.0073,  0.0222,  0.0130],
        [ 0.0035,  0.0300, -0.0216,  ...,  0.0047,  0.0302,  0.0219],
        [ 0.0126, -0.0044, -0.0125,  ..., -0.0150,  0.0049,  0.0061],
        ...,
        [-0.0108,  0.0126, -0.0327,  ..., -0.0164,  0.0123,  0.0289],
        [ 0.0290, -0.0191, -0.0046,  ...,  0.0155, -0.0124, -0.0242],
        [ 0.0268,  0.0019,  0.0008,  ...,  0.0200, -0.0121,  0.0170]],
       requires_grad=True))
('model.0.bias', Parameter containing:
tensor([ 0.0161,  0.0350, -0.0285,  0.0142,  0.0265,  0.0028, -0.0252,  0.0126,
        -0.0002,  0.0255,  0.0119,  0.0272,  0.0286,  0.0141,  0.0332,  0.0306,
         0.0011, -0.0128, -0.0094, -0.0159,  0.0252, -0.0331, -0.0063, -0.0230,
        -0.0002, -0.0156,  0.0014,  0.0088, -0.0329, -0.0151, -0.0186, -0.0200,
         0.0302, -0.0018,  0.0128,  0.0237,  0.0005, -0.0244,  0.0094,  0.0076,
         0.0353, -0.0197, -0.0133,  0.0354, -0.0322, -0.00

In [57]:
device

device(type='cuda')

In [58]:
# making model use gpu instead of cpu
model = model.to(device)

In [59]:
model

ImageClassifier(
  (model): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.4, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.4, inplace=False)
    (8): Linear(in_features=64, out_features=32, bias=True)
    (9): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): Dropout(p=0.4, inplace=False)
    (12): Linear(in_features=32, out_features=10, bias=True)
  )
)

In [60]:
epochs = 30
total_loss = 0
for epoch in range(epochs):
    epoch += 1
    total_loss = 0
    for features_batch, targets_batch in train_dataset_loader:
        features_batch = features_batch.to(device)
        targets_batch = targets_batch.to(device)
        y_pred = model(features_batch)
        loss = model.loss_function(targets_batch)
        model.optimization()
        total_loss = total_loss + loss

    print("epoch:", epoch, "\tloss:", total_loss/len(train_dataset_loader))

epoch: 1 	loss: 0.8855716113448143
epoch: 2 	loss: 0.6839047253529231
epoch: 3 	loss: 0.62979975532492
epoch: 4 	loss: 0.6036474624276161
epoch: 5 	loss: 0.5904092254340648
epoch: 6 	loss: 0.5646680973271528
epoch: 7 	loss: 0.5618921341598034
epoch: 8 	loss: 0.5457620586206515
epoch: 9 	loss: 0.5377527421712875
epoch: 10 	loss: 0.5338332143227259
epoch: 11 	loss: 0.5253183320661385
epoch: 12 	loss: 0.5184106257657211
epoch: 13 	loss: 0.5117883075674375
epoch: 14 	loss: 0.5070243808825811
epoch: 15 	loss: 0.5084956451455752
epoch: 16 	loss: 0.5021026213268439
epoch: 17 	loss: 0.5045961494247119
epoch: 18 	loss: 0.49281077496210735
epoch: 19 	loss: 0.49758934845527014
epoch: 20 	loss: 0.49030053212245306
epoch: 21 	loss: 0.4901267775396506
epoch: 22 	loss: 0.48743234515190126
epoch: 23 	loss: 0.48346531042456625
epoch: 24 	loss: 0.4780094075202942
epoch: 25 	loss: 0.4758041877498229
epoch: 26 	loss: 0.4805025438070297
epoch: 27 	loss: 0.4702957763671875
epoch: 28 	loss: 0.468754907280206

In [61]:
# model ready for evaluation
model.eval()

ImageClassifier(
  (model): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.4, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.4, inplace=False)
    (8): Linear(in_features=64, out_features=32, bias=True)
    (9): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): Dropout(p=0.4, inplace=False)
    (12): Linear(in_features=32, out_features=10, bias=True)
  )
)

### Accuracy:
We need to find the accurcy now. The accuracy of the data is given by the formula:

Accuracy = (number of correct predictions) / (total number of the predictions)

In [62]:
len(test_dataset_loader)*32

12000

In [63]:
correct = 0
data_records = 0
with torch.no_grad():
    for features_batch, targets_batch in test_dataset_loader:
        features_batch = features_batch.to(device)
        targets_batch = targets_batch.to(device)
        y_pred = model(features_batch)
        correct = correct + (targets_batch == torch.max(y_pred, dim = 1)[1]).sum().item()
        data_records = data_records + features_batch.shape[0]
    print("total data_records:", data_records, "correct records:", correct)
    accuracy = correct / data_records
    print(accuracy)

total data_records: 12000 correct records: 10521
0.87675


In [64]:
correct = 0
data_records = 0
with torch.no_grad():
    for features_batch, targets_batch in train_dataset_loader:
        features_batch = features_batch.to(device)
        targets_batch = targets_batch.to(device)
        y_pred = model(features_batch)
        correct = correct + (targets_batch == torch.max(y_pred, dim = 1)[1]).sum().item()
        data_records = data_records + features_batch.shape[0]
    print("total data_records:", data_records, "correct records:", correct)
    accuracy = correct / data_records
    print(accuracy)

total data_records: 48000 correct records: 43186
0.8997083333333333


## Hyperparameter Tuning using Optuna:
Number of layers

Number of hidden neurons

numer of epochs

optimizer

learning rate

batch size

dropout

weight decay

In [ ]:
# defining the class for ImgaeClassifier but in a different way:
# for optuna
class ImageClassifier(nn.Module):

    def __init__(self, no_of_features, no_of_outputs, no_of_hidden_layers, neurons_per_layer, dropout_rate):
        super().__init__()

        layers = []

        for i in range(no_of_hidden_layers):
            layers.append(nn.Linear(no_of_features, neurons_per_layer))
            layers.append(nn.BatchNorm1d(neurons_per_layer))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            no_of_features = neurons_per_layer
        layers.append(nn.Linear(neurons_per_layer, no_of_outputs))

        # model
        self.model = nn.Sequential(*layers)
    
    def forward(self, features):
        return self.model(features)

### defining the objective function

search space

model init

parameter init

training loop

evaluation loop

In [ ]:
# defining the objective function

def objective(trial):
    # next hyperparameter value from the search space
    number_of_hidden_layers = trial.suggest_int("num_hidden_layers", 1, 5)
    number_of_neurons = trial.suggest_int("num_of_neurons", 8, 128, step=8)
    epochs = trial.suggest_int("epochs", 20, 50, step = 10)
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-1, log = True)
    dropout_rate = trial.suggest_float("droput_rate", 0.1, 0.5, step = 0.1)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64, 128])
    optimizer_name = trial.suggest_categorical("optimizer", ['Adam', 'SGD', 'RMSprop'])
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-3, log = True)

    # dataset batch selection
    train_dataset_loader = DataLoader(train_dataset, batch_size, shuffle= True, pin_memory=True)
    test_dataset_loader = DataLoader(test_dataset, batch_size, shuffle=False, pin_memory=True) 

    # model init
    input_dim = 784
    output_dim = 10

    model = ImageClassifier(input_dim, output_dim, number_of_hidden_layers, number_of_neurons, dropout_rate) #cpu based model
    model.to(device) #gpu based model

    # loss function
    loss_fn = nn.CrossEntropyLoss()
    # optimization + l2 regularization
    if optimizer_name == "SGD":
        optim = torch.optim.SGD(model.parameters(), lr= learning_rate, weight_decay= weight_decay)
    elif optimizer_name == "Adam":
        optim = torch.optim.Adam(model.parameters(), lr= learning_rate, weight_decay= weight_decay)
    else:
        optim = torch.optim.RMSprop(model.parameters(), lr= learning_rate, weight_decay=weight_decay)

    # training loop
    for epoch in range(epochs):
        epoch += 1
        for features_batch, targets_batch in train_dataset_loader:
            # change the cpu data to gpu
            features_batch = features_batch.to(device)
            targets_batch = targets_batch.to(device)
            # forward pass
            y_pred = model(features_batch)
            # find the loss
            loss = loss_fn(y_pred, targets_batch)
            # remove all the gradients
            optim.zero_grad()
            # gradient
            loss.backward()
            # update the weight and bias
            optim.step()

    # evaluation loop
    model.eval()
    correct = 0
    data_records = 0
    with torch.no_grad():
        for features_batch, targets_batch in test_dataset_loader:
            features_batch = features_batch.to(device)
            targets_batch = targets_batch.to(device)
            y_pred = model(features_batch)
            correct = correct + (targets_batch == torch.max(y_pred, dim = 1)[1]).sum().item()
            data_records = data_records + features_batch.shape[0]
        accuracy = correct / data_records

    # return accruacy
    return accuracy

In [ ]:
study = optuna.create_study(direction="maximize")

In [ ]:
study.optimize(objective, n_trials=10)

In [ ]:
study.best_params()